# Modelo que define quando acionar a bomba


In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix


df = pd.read_csv("../data/processed/linha3_10min.csv", parse_dates=["t"])

print("tamanho da tabela:", df.shape)
print(df.head())

tamanho da tabela: (26928, 17)
                    t   ano  hora_local   fase  umid_solo_15cm  \
0 2024-06-12 00:00:00  2024           2  outra             NaN   
1 2024-06-12 00:10:00  2024           2  outra             NaN   
2 2024-06-12 00:20:00  2024           2  outra             NaN   
3 2024-06-12 00:30:00  2024           2  outra             NaN   
4 2024-06-12 00:40:00  2024           2  outra             NaN   

   umid_solo_30cm  u_15cm  u_30cm  temp_solo_15cm  temp_solo_30cm  temp_ar  \
0             NaN     NaN     NaN            24.7            24.2     17.6   
1             NaN     NaN     NaN            24.7            24.2     17.5   
2             NaN     NaN     NaN            24.7            24.2     17.3   
3             NaN     NaN     NaN            24.7            24.2     17.2   
4             NaN     NaN     NaN            24.6            24.2     17.2   

   umid_ar  vpd_kpa  valvula  litros  chuva_suspeita  buraco  
0     81.0    0.382    False     0.0    

In [3]:

partes = []
for ano, d in df.groupby("ano"):
    d = d.copy()
    inicio = d["valvula"] & ~d["valvula"].shift(1, fill_value=False)
    
    proximas = [inicio.shift(-k, fill_value=False) for k in range(1, 7)]
    d["alvo"] = pd.concat(proximas, axis=1).any(axis=1).astype(int)
    
    d["vpd_3h"] = d["vpd_kpa"].rolling(18, min_periods=1).mean()
    partes.append(d)
df = pd.concat(partes)

print("contagem do alvo (0 = não regar, 1 = regar):")
print(df["alvo"].value_counts())
print("linhas com alvo = 1 por ano:")
print(df.groupby("ano")["alvo"].sum())

contagem do alvo (0 = não regar, 1 = regar):
alvo
0    26276
1      652
Name: count, dtype: int64
linhas com alvo = 1 por ano:
ano
2024    305
2025    347
Name: alvo, dtype: int64


In [4]:
df = df[(df["fase"] == "controle") & ~df["buraco"] & ~df["chuva_suspeita"] & ~df["valvula"]]

entradas = ["u_15cm", "vpd_kpa", "vpd_3h"]

print("linhas depois do filtro:", len(df))
print(df.groupby("ano").size())
print(df[entradas].describe().round(2))

linhas depois do filtro: 7232
ano
2024    3853
2025    3379
dtype: int64
        u_15cm  vpd_kpa   vpd_3h
count  7232.00  7232.00  7232.00
mean      0.50     1.52     1.54
std       0.44     1.22     1.19
min      -0.78     0.00     0.00
25%       0.18     0.55     0.59
50%       0.42     1.08     1.12
75%       0.70     2.29     2.36
max       3.06     5.20     5.14


In [5]:
treino, teste = df[df["ano"] == 2024], df[df["ano"] == 2025]
X_treino, y_treino = treino[entradas], treino["alvo"]
X_teste, y_teste = teste[entradas], teste["alvo"]
print(f"treino: {len(treino)} linhas ({y_treino.mean():.1%} com rega na próxima hora)")
print(f"teste:  {len(teste)} linhas ({y_teste.mean():.1%} com rega na próxima hora)")
print("formato X_treino:", X_treino.shape, "| formato X_teste:", X_teste.shape)

treino: 3853 linhas (5.8% com rega na próxima hora)
teste:  3379 linhas (9.2% com rega na próxima hora)
formato X_treino: (3853, 3) | formato X_teste: (3379, 3)


In [6]:
modelo = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced"))
modelo.fit(X_treino, y_treino)

print("modelo treinado:", modelo)
print("acurácia no treino:", round(modelo.score(X_treino, y_treino), 3))

modelo treinado: Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(class_weight='balanced'))])
acurácia no treino: 0.892


In [7]:
y_prev = modelo.predict(X_teste)
print("\nMatriz de confusão (linhas = real, colunas = previsto):")
print(confusion_matrix(y_teste, y_prev))
print(classification_report(y_teste, y_prev, target_names=["não regar", "regar"]))

print("vezes que o modelo disse 'regar':", y_prev.sum(), "| vezes reais:", y_teste.sum())


Matriz de confusão (linhas = real, colunas = previsto):
[[2875  193]
 [  21  290]]
              precision    recall  f1-score   support

   não regar       0.99      0.94      0.96      3068
       regar       0.60      0.93      0.73       311

    accuracy                           0.94      3379
   macro avg       0.80      0.93      0.85      3379
weighted avg       0.96      0.94      0.94      3379

vezes que o modelo disse 'regar': 483 | vezes reais: 311


In [ ]:
coefs = modelo.named_steps["logisticregression"].coef_[0]
print(pd.Series(coefs, index=entradas).round(2))

print("intercepto:", round(modelo.named_steps["logisticregression"].intercept_[0], 2))